<a href="https://colab.research.google.com/github/bhaskarpondugala/NASSCOM_AI_FDP/blob/main/Day10/U22_Kafka_Spark_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U22 · Big Data: Kafka & Spark — Hands-on

**Idea of this notebook:** when data outgrows one machine you *scale out* across a cluster —
**Kafka** moves the data, **Spark** crunches it. Real Kafka/Spark need a cluster, so here we
build tiny pure-Python models of their core ideas so you can run and *see* them.

Small practicals:
1. Divide & conquer — partition data, run a task per partition, reduce (distributed compute)
2. Mini-Kafka — topics, partitions, offsets, partition-by-key (order per key)
3. Consumer groups & offsets — resume, independent groups, replay
4. Spark-style lazy evaluation — transformations (lazy) vs actions (trigger) + the plan/DAG
5. Capstone — a Kafka → Spark → sink micro-batch pipeline

Every line of code is commented. Run the cells top to bottom.

### Setup — standard library only

In [1]:
from collections import Counter          # Counter: easy word/键 counting and merging
from concurrent.futures import ThreadPoolExecutor  # run our partition "tasks" together
import zlib                                  # crc32: a stable hash for partition assignment

## 1. Divide & conquer (distributed computing basics)

A big job is split into chunks (**partitions**); each chunk is processed by its own **task**
(in parallel on a real cluster); the partial results are then **reduced** into one answer.
We verify the distributed result equals the single-machine result.

In [2]:
# Synthetic "log lines" — pretend this is a large dataset of events
lines = ["error db timeout", "ok request", "error db timeout", "ok cache",
         "warn slow query", "ok request", "error network", "ok request"] * 50   # 400 lines

# --- Single machine: one pass over everything ---
single = Counter()                                  # one global word counter
for ln in lines:                                    # walk every line
    single.update(ln.split())                       # count each word

# --- Distributed: partition -> map each partition (a task) -> reduce ---
def partition(data, k):                             # split data into k roughly-equal chunks
    size = (len(data) + k - 1) // k                 # chunk size (round up)
    return [data[i:i+size] for i in range(0, len(data), size)]  # list of chunks

def map_task(chunk):                                # the work done on ONE partition
    c = Counter()                                   # a local counter for this chunk
    for ln in chunk:                                # only this chunk's lines
        c.update(ln.split())
    return c                                         # return the partial result

parts = partition(lines, 4)                         # 4 partitions -> 4 tasks
with ThreadPoolExecutor(max_workers=4) as pool:     # run the tasks together
    partials = list(pool.map(map_task, parts))      # one partial Counter per partition

distributed = Counter()                             # REDUCE: merge the partials into one
for p in partials:
    distributed.update(p)

print("Partition sizes      :", [len(p) for p in parts])      # the work was split 4 ways
print("Single-machine result:", single.most_common(3))
print("Distributed   result:", distributed.most_common(3))
print("Identical?           :", single == distributed)        # same answer, computed in parallel

Partition sizes      : [100, 100, 100, 100]
Single-machine result: [('ok', 200), ('error', 150), ('request', 150)]
Distributed   result: [('ok', 200), ('error', 150), ('request', 150)]
Identical?           : True


## 2. Mini-Kafka (topics, partitions, offsets)

Kafka is a distributed, append-only **log**. A **topic** is split into **partitions**; each event
gets an **offset** (its position). The partition is chosen from the event **key**, so all events
for one key land in the same partition — and order is guaranteed *within* a partition.

In [3]:
class MiniKafka:
    """A tiny in-memory Kafka: a topic split into ordered, append-only partitions."""
    def __init__(self, n_partitions):
        self.n = n_partitions                       # how many partitions this topic has
        self.log = {p: [] for p in range(n_partitions)}  # partition -> list of (offset, key, value)

    def _partition_for(self, key):                  # map a key to a partition (deterministic)
        return zlib.crc32(key.encode()) % self.n    # same key -> always the same partition

    def produce(self, key, value):                  # append one event to the right partition
        p = self._partition_for(key)                # pick the partition from the key
        offset = len(self.log[p])                    # offset = next position in that partition
        self.log[p].append((offset, key, value))     # append (the log only ever grows)
        return p, offset                             # tell the producer where it landed

    def show(self):                                  # print the topic like the slide diagram
        for p in range(self.n):
            evs = "  ".join(f"[{o}:{k}={v}]" for o, k, v in self.log[p])  # events in offset order
            print(f"  P{p}: {evs}")

topic = MiniKafka(n_partitions=3)                   # create topic 'orders' with 3 partitions
for key, val in [("leo","ord1"), ("ken","ord2"), ("leo","ord3"),
                 ("alice","ord4"), ("ken","ord5"), ("leo","ord6")]:
    p, o = topic.produce(key, val)                   # publish each event
    print(f"produced {key}={val}  -> partition {p}, offset {o}")

print("\nTopic 'orders' (offset grows left -> right):")
topic.show()
leo_p = topic._partition_for("leo")                 # where do leo's events live?
print(f"\nAll 'leo' events are in P{leo_p}, in produce order — order preserved per key.")

produced leo=ord1  -> partition 0, offset 0
produced ken=ord2  -> partition 1, offset 0
produced leo=ord3  -> partition 0, offset 1
produced alice=ord4  -> partition 2, offset 0
produced ken=ord5  -> partition 1, offset 1
produced leo=ord6  -> partition 0, offset 2

Topic 'orders' (offset grows left -> right):
  P0: [0:leo=ord1]  [1:leo=ord3]  [2:leo=ord6]
  P1: [0:ken=ord2]  [1:ken=ord5]
  P2: [0:alice=ord4]

All 'leo' events are in P0, in produce order — order preserved per key.


## 3. Consumer groups & offsets (resume + replay)

Each **consumer group** tracks its own **offset** per partition. After a restart it resumes from
the last committed offset (no reprocessing). A *different* group reads independently. And because
Kafka keeps the events, you can **replay** by resetting the offset to 0.

In [4]:
committed = {}                                      # (group, partition) -> next offset to read

def consume(topic, group, partition, max_events=100):
    start = committed.get((group, partition), 0)    # resume from this group's last position
    events = topic.log[partition][start:start + max_events]  # read a slice of the log
    if events:                                       # if we read anything...
        committed[(group, partition)] = events[-1][0] + 1   # ...commit: next offset after last
    return [(o, k, v) for o, k, v in events]         # return what we read

print("billing reads P0          :", consume(topic, "billing", 0))     # group 'billing' reads all
print("billing reads P0 again    :", consume(topic, "billing", 0))     # resumes -> nothing new
print("analytics reads P0        :", consume(topic, "analytics", 0))   # different group, from 0
committed[("billing", 0)] = 0                        # RESET billing's offset (replay)
print("billing REPLAY P0 from 0  :", consume(topic, "billing", 0))     # reprocess history

billing reads P0          : [(0, 'leo', 'ord1'), (1, 'leo', 'ord3'), (2, 'leo', 'ord6')]
billing reads P0 again    : []
analytics reads P0        : [(0, 'leo', 'ord1'), (1, 'leo', 'ord3'), (2, 'leo', 'ord6')]
billing REPLAY P0 from 0  : [(0, 'leo', 'ord1'), (1, 'leo', 'ord3'), (2, 'leo', 'ord6')]


## 4. Spark-style lazy evaluation

In Spark, **transformations** (filter, select, groupBy) are *lazy* — they only build a plan (a
DAG). An **action** (count, collect, show, write) triggers the whole plan at once, so Spark can
optimise it first. We build a toy `LazyFrame` to feel the difference.

In [5]:
class LazyFrame:
    """A toy Spark-style DataFrame: transformations are LAZY, actions TRIGGER execution."""
    def __init__(self, rows, plan=None):
        self.rows = rows                            # source data (list of dict rows)
        self.plan = plan or []                      # the recorded transformations (the DAG)

    # --- transformations: lazy. Return a NEW LazyFrame, run nothing. ---
    def filter(self, fn, desc):
        return LazyFrame(self.rows, self.plan + [("filter", fn, desc)])      # append to the plan
    def groupby_sum(self, key, value):
        d = f"groupBy {key} -> sum({value})"
        return LazyFrame(self.rows, self.plan + [("groupby_sum", (key, value), d)])

    def explain(self):                              # show the plan WITHOUT executing it
        print("Plan (lazy DAG):")
        for i, (_op, _arg, desc) in enumerate(self.plan, 1):
            print(f"  {i}. {desc}")

    def _run(self):                                 # execute every step in order
        data = self.rows
        for op, arg, _desc in self.plan:
            if op == "filter":                      # keep rows matching the predicate
                data = [r for r in data if arg(r)]
            elif op == "groupby_sum":               # sum `value` grouped by `key`
                key, value = arg
                agg = {}
                for r in data:
                    agg[r[key]] = agg.get(r[key], 0) + r[value]
                data = [{key: k, value: v} for k, v in agg.items()]
        return data

    # --- actions: trigger. They run the whole plan. ---
    def collect(self):
        print(">> action collect() — executing the plan now")
        return self._run()
    def count(self):
        print(">> action count() — executing the plan now")
        return len(self._run())

orders = [                                          # tiny synthetic dataset
    {"region": "North", "status": "paid",   "amount": 100},
    {"region": "South", "status": "paid",   "amount": 200},
    {"region": "North", "status": "unpaid", "amount": 50},
    {"region": "South", "status": "paid",   "amount": 300},
]
df = LazyFrame(orders)                              # wrap the data
daily = (df.filter(lambda r: r["status"] == "paid", "keep status == 'paid'")  # lazy
            .groupby_sum("region", "amount"))                                 # lazy
print("Built the pipeline — but nothing has run yet (lazy).\n")
daily.explain()                                     # inspect the DAG
print()
print("Result:", daily.collect())                   # the ACTION triggers execution

Built the pipeline — but nothing has run yet (lazy).

Plan (lazy DAG):
  1. keep status == 'paid'
  2. groupBy region -> sum(amount)

>> action collect() — executing the plan now
Result: [{'region': 'North', 'amount': 100}, {'region': 'South', 'amount': 500}]


## 5. Capstone — a Kafka → Spark → sink micro-batch pipeline

The real-time pattern: **Kafka** buffers bursty events; **Spark Structured Streaming** reads
**micro-batches**, aggregates, and writes to a **sink**. We combine Parts 2–4: events land in
mini-Kafka, each micro-batch reads only the *new* events and updates a running revenue table.

In [6]:
stream = MiniKafka(n_partitions=1)                  # one-partition topic for simplicity
sink = {}                                           # our "warehouse": running revenue by region

def micro_batch(topic, group="streamer", partition=0):
    events = consume(topic, group, partition)       # read ONLY the new events since last offset
    rows = [{"region": k.split(":")[0], "amount": int(v)} for o, k, v in events]  # parse events
    if not rows:                                    # nothing new this batch
        print("  (no new events)")
        return
    agg = LazyFrame(rows).groupby_sum("region", "amount").collect()  # Spark-style aggregation
    for r in agg:                                   # update the sink with this batch's totals
        sink[r["region"]] = sink.get(r["region"], 0) + r["amount"]

# Producers send a first burst of events (key = "region:customer", value = amount)
for k, v in [("North:c1", "100"), ("South:c2", "200"), ("North:c3", "50")]:
    stream.produce(k, v)
print("Micro-batch 1:")
micro_batch(stream)                                 # process the new events
print("  sink:", sink)

# More events arrive between batches
for k, v in [("South:c4", "300"), ("North:c5", "80")]:
    stream.produce(k, v)
print("Micro-batch 2:")
micro_batch(stream)                                 # processes ONLY the 2 new events
print("  sink:", sink)

Micro-batch 1:
>> action collect() — executing the plan now
  sink: {'North': 150, 'South': 200}
Micro-batch 2:
>> action collect() — executing the plan now
  sink: {'North': 230, 'South': 500}


## Wrap-up — the big-data backbone

You modelled the two pillars from the slides: **Kafka** (a partitioned, replay-able event log)
and **Spark** (lazy, partition-parallel compute), then wired them into a streaming pipeline.

**Takeaways**
- Scale *out*: split data into partitions, run a task per partition, reduce (Part 1).
- A topic = ordered partitions; the key picks the partition; offsets index events (Part 2).
- Consumer groups track offsets independently — resume after restart, or replay from 0 (Part 3).
- Transformations are lazy; an action triggers the optimised plan (Part 4).
- Kafka buffers, Spark reads micro-batches and writes to a sink (Part 5).

**Real tools to graduate to:** `kafka-python` / a local Kafka broker, and `pyspark`
(`SparkSession`, DataFrames, `readStream`/`writeStream`). Reach for a cluster only when the data
genuinely needs it — small data is happier in pandas or DuckDB.

**Try yourself:** give the mini-Kafka topic 3 partitions in Part 5 and run a micro-batch per
partition — that's exactly how Spark parallelises a Kafka source.